# Đánh Giá (Validation) Mô Hình MulCo
Notebook này nạp cấu trúc mô hình, tải trọng số đã huấn luyện và tiến hành đánh giá trên tập Validation.

In [1]:
import os
import sys
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import CLIPTokenizer, CLIPModel
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

current_dir = Path.cwd()
PROJECT_ROOT = current_dir
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.datasets.multimodal_raw_dataset import MultiModalRawDataset
from src.models.backbones.vision.convnext_cbam import ConvNeXt_CBAM
from src.models.fusion.mulco_fusion import MulCoFusionBlock
from src.models.multimodal.mulco_classifier import Conv1x1Classifier

In [2]:
class MulCoEndToEnd(nn.Module):
    def __init__(self, num_classes=28, proj_dim=512, spatial_size=(7, 7)):
        super().__init__()
        self.image_backbone = ConvNeXt_CBAM(num_classes=num_classes)
        self.text_backbone = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").text_model
        self.img_proj = nn.Conv2d(1024, proj_dim, kernel_size=1)
        self.txt_proj = nn.Linear(512, proj_dim)
        self.fusion_blocks = nn.ModuleList([MulCoFusionBlock(dim=proj_dim, num_heads=8) for _ in range(3)])
        self.classifier = Conv1x1Classifier(in_channels=proj_dim, num_classes=num_classes, spatial_size=spatial_size)

    def forward(self, images, input_ids, attention_mask):
        img_feat = self.image_backbone.forward_features_spatial(images)
        txt_out = self.text_backbone(input_ids=input_ids, attention_mask=attention_mask)
        txt_feat = txt_out.last_hidden_state
        img_feat = self.img_proj(img_feat)
        txt_feat = self.txt_proj(txt_feat)
        
        for block in self.fusion_blocks:
            img_feat, txt_feat = block(img_feat, txt_feat)
            
        return self.classifier(img_feat)

In [3]:
def custom_collate_fn(batch, tokenizer):
    images = torch.stack([b["image"] for b in batch])
    labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    texts = [b["text"] for b in batch]
    text_tokens = tokenizer(texts, padding=True, truncation=True, max_length=77, return_tensors="pt")
    return images, text_tokens.input_ids, text_tokens.attention_mask, labels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

val_dataset = MultiModalRawDataset(
    image_root=os.path.join(PROJECT_ROOT, "data/AIDG/dataset_PlantDoc/images/val"),
    caption_root=os.path.join(PROJECT_ROOT, "data/AIDG/captions_LLaVA/val"),
    transform=transform,
    use_depth_suppressed=False,
    strict_caption_match=False
)

val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False, num_workers=2,
    collate_fn=lambda b: custom_collate_fn(b, tokenizer)
)

model = MulCoEndToEnd(num_classes=28).to(device)
ckpt_path = os.path.join(PROJECT_ROOT, "archive", "cross_attention_3blocks", "best_model.pth")
model.load_state_dict(torch.load(ckpt_path))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, input_ids, attn_mask, labels in tqdm(val_loader, desc="Validating"):
        images, input_ids, attn_mask = images.to(device), input_ids.to(device), attn_mask.to(device)
        logits = model(images, input_ids, attn_mask)
        preds = torch.argmax(logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)

print(f"\nAccuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

[MultiModalRawDataset] Total selected images: 635
[MultiModalRawDataset] Valid samples: 635
[MultiModalRawDataset] Skipped missing caption: 0
[MultiModalRawDataset] Skipped invalid caption: 0
[MultiModalRawDataset] Matched by external mapping: 0
[MultiModalRawDataset] Num classes: 28
[MultiModalRawDataset] class_to_idx: {'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3, 'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7, 'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11, 'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14, 'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_leaf': 17, 'Tomato_Septoria_leaf_spot': 18, 'Tomato_leaf': 19, 'Tomato_leaf_bacterial_spot': 20, 'Tomato_leaf_late_blight': 21, 'Tomato_leaf_mosaic_virus': 22, 'Tomato_leaf_yellow_virus': 23, 'Tomato_mold_leaf': 24, 'Tomato_two_spotted_spider_mites_leaf'

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Validating:   0%|          | 0/40 [00:00<?, ?it/s]


Accuracy : 0.9953
Precision: 0.9955
Recall   : 0.9953
F1-Score : 0.9953
